# 🎮 GTA 6 Viral Clipper — Google Colab Edition
Gere cortes verticais virais automáticos para Reels, Shorts e TikTok direto na nuvem do Google!

### ⚡ Benefícios no Google Colab:
- **GPU Grátis (NVIDIA T4):** Transcrição com Whisper até 10x mais rápida do que no computador local.
- **Internet Gigabit:** Downloads parciais ultrarrápidos direto dos servidores do YouTube/Twitch.
- **Zero consumo local:** Não esquenta seu PC, não usa memória nem espaço em disco local.
- **Google Drive Integrado:** Todos os vídeos são salvos diretamente na sua conta do Google Drive.

In [ ]:
#@title 📁 1. Conectar ao Google Drive
#@markdown Clique no Play. Se abrir a janela de permissão, selecione sua conta Google e clique em Permitir.
import os
try:
    from google.colab import auth
    auth.authenticate_user()
except Exception as e:
    print('Aviso auth:', e)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('✓ Google Drive montado com sucesso!')
except Exception as e:
    print('Erro ao montar:', e)
    print('DICA: Se der erro de credencial, clique no ícone de Pasta (Arquivos) na barra lateral esquerda e depois no botão com o logo do Google Drive para montar graficamente!')

In [ ]:
#@title ⚙️ 2. Instalar Dependências & Atualizar Projeto
#@markdown Instala FFmpeg, fontes Montserrat, bibliotecas CUDA 12, detector facial YuNet e sincroniza o código.
%cd /content
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-montserrat
!pip install -q -U --pre "yt-dlp[default]"
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12
!pip install -q faster-whisper opencv-python-headless numpy certifi static-ffmpeg deno

# Baixar modelo de deteccao de rosto YuNet para Split-Screen
!mkdir -p /root/clipper-gta6/modelos
!curl -fsSL -o /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx

!rm -rf /content/meu-gta6-clipper
!git clone -b turbo-v2 https://github.com/papaigato/meu-gta6-clipper.git /content/meu-gta6-clipper
%cd /content/meu-gta6-clipper
print('✓ Ambiente configurado e 100% atualizado (branch turbo-v2)!')

In [ ]:
#@title 🍪 2.1 (Opcional) Subir cookies do YouTube
#@markdown Se o YouTube bloquear o IP do Colab com pedido de login, suba seu arquivo cookies.txt aqui (ele fica salvo no seu Google Drive). Marque **SUBSTITUIR_COOKIES** se quiser trocar um cookie antigo/expirado:
SUBSTITUIR_COOKIES = False #@param {type:"boolean"}
import shutil
from pathlib import Path

drive_dir = Path('/content/drive/MyDrive')
drive_cookie = drive_dir / 'cookies.txt'
local_cookie = Path('/content/cookies.txt')

if drive_cookie.exists() and drive_cookie.stat().st_size > 0 and not SUBSTITUIR_COOKIES:
    shutil.copyfile(drive_cookie, local_cookie)
    print('✓ Cookies encontrados no Google Drive e carregados! (Marque SUBSTITUIR_COOKIES se quiser enviar um novo)')
else:
    try:
        from google.colab import files
        print('Clique em "Escolher arquivos" abaixo e selecione o seu cookies.txt:')
        uploaded = files.upload()
        if not uploaded:
            print('Nenhum arquivo selecionado.')
        for fn, data in uploaded.items():
            local_cookie.write_bytes(data)
            if drive_dir.exists():
                drive_cookie.write_bytes(data)
                print(f'✓ Cookies ({fn}) salvos em /content/drive/MyDrive/cookies.txt e /content/cookies.txt!')
            else:
                print(f'✓ Cookies ({fn}) salvos em /content/cookies.txt! (Drive não montado)')
            break
    except Exception as e:
        print('Erro no upload:', e)
        print('Dica: você também pode arrastar o arquivo cookies.txt direto para a pasta /content/ na barra lateral esquerda.')

In [ ]:
#@title 🎬 3. Gerar Cortes da Live
#@markdown Escolha o link da live, a quantidade, o estilo e a qualidade máxima de download:

LINK_DA_LIVE = "https://www.youtube.com/watch?v=X2rtbJJEvVE" #@param {type:"string"}
QUANTIDADE_DE_CORTES = 30 #@param {type:"integer"}
RESOLUCAO_DOWNLOAD = "1440p60 (Ultra-HD 2K - Maxima Qualidade)" #@param ["1440p60 (Ultra-HD 2K - Maxima Qualidade)", "1080p60 (Full HD)", "720p60 (Mais rapido)"]
ESTILO_VISUAL = "Split-Screen 30/70 (TikTok Grota Clips: Webcam Topo + Jogo)" #@param ["Split-Screen 30/70 (TikTok Grota Clips: Webcam Topo + Jogo)", "Fundo Desfocado (Blur - Estilo Video 8)"]
REFAZER_CORTES_EXISTENTES = False #@param {type:"boolean"}
PASTA_DESTINO_DRIVE = "/content/drive/MyDrive/GTA6_Cortes" #@param {type:"string"}

import os, glob, sys
# Garante detector de rosto baixado no Colab
!mkdir -p /root/clipper-gta6/modelos
![ ! -f /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx ] && curl -fsSL -o /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx

# Sempre sincroniza a versao mais recente da branch turbo-v2
if not os.path.exists('/content/meu-gta6-clipper'):
    !git clone -b turbo-v2 https://github.com/papaigato/meu-gta6-clipper.git /content/meu-gta6-clipper
else:
    !git -C /content/meu-gta6-clipper fetch origin
    !git -C /content/meu-gta6-clipper checkout turbo-v2
    !git -C /content/meu-gta6-clipper pull origin turbo-v2
%cd /content/meu-gta6-clipper

# Garante bibliotecas CUDA 12 (cuBLAS/cuDNN) no LD_LIBRARY_PATH para Whisper rodar na GPU T4
nv_libs = [p for base in sys.path for p in glob.glob(os.path.join(base, 'nvidia', '*', 'lib')) if os.path.isdir(p)]
if nv_libs:
    atual_ld = os.environ.get('LD_LIBRARY_PATH', '')
    os.environ['LD_LIBRARY_PATH'] = ':'.join(dict.fromkeys(nv_libs + ([atual_ld] if atual_ld else [])))

# Salva cache de lives (transcricao, escolhidos) no Google Drive e arquivos pesados temporarios no SSD local (/tmp)
CACHE_DRIVE = f"{PASTA_DESTINO_DRIVE}/.cache"
!mkdir -p "$CACHE_DRIVE/modelos" /tmp/clipper_gta6
![ -f /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx ] && cp -u /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx "$CACHE_DRIVE/modelos/"

if REFAZER_CORTES_EXISTENTES:
    # Remove pedacos antigos para forcar o download limpo
    !rm -rf /tmp/clipper_gta6/*
    !rm -f /root/clipper-gta6/trabalho/*/*/trecho.* /root/clipper-gta6/trabalho/*/*/*.webm /root/clipper-gta6/trabalho/*/*/*.mkv /root/clipper-gta6/trabalho/*/*/feito.txt
    !rm -f "$CACHE_DRIVE/trabalho"/*/*/trecho.* "$CACHE_DRIVE/trabalho"/*/*/*.webm "$CACHE_DRIVE/trabalho"/*/*/*.mkv "$CACHE_DRIVE/trabalho"/*/*/feito.txt

max_res = "1440" if "1440" in RESOLUCAO_DOWNLOAD else ("1080" if "1080" in RESOLUCAO_DOWNLOAD else "720")
os.environ["CLIPPER_MAX_RES"] = max_res
os.environ["CLIPPER_CORTES_DIR"] = PASTA_DESTINO_DRIVE
os.environ["CLIPPER_GTA6_HOME"] = CACHE_DRIVE
os.environ["CLIPPER_TMP_DIR"] = "/tmp/clipper_gta6"
os.environ["CLIPPER_ESTILO"] = "blur" if "Blur" in ESTILO_VISUAL else "split"
os.environ["CLIPPER_FORCAR_REFAZER"] = "1" if REFAZER_CORTES_EXISTENTES else "0"
os.environ["CLIPPER_ESPERA"] = "86400"

print(f"🚀 Link da Live: {LINK_DA_LIVE}")
print(f"💎 Resolucao de Download: {max_res}p60 (Ultra-HD)")
print(f"🎨 Estilo: {os.environ['CLIPPER_ESTILO']}")
print(f"♻️ Refazer cortes antigos com download limpo: {REFAZER_CORTES_EXISTENTES}")
print(f"📁 Destino: {PASTA_DESTINO_DRIVE}")
print(f"💾 Cache permanente no Drive: {CACHE_DRIVE}")

!python /content/meu-gta6-clipper/scripts/clip.py automatico "$LINK_DA_LIVE" --quantos $QUANTIDADE_DE_CORTES


In [ ]:
#@title 📋 4. Verificar Cortes Salvos no Google Drive
import glob, os
videos = glob.glob(f"{PASTA_DESTINO_DRIVE}/*/*.mp4")
print(f"🎬 Total de vídeos prontos no Google Drive: {len(videos)}")
for v in sorted(videos)[-15:]:
    mb = os.path.getsize(v) / (1024 * 1024)
    print(f"✓ {os.path.basename(v)} ({mb:.1f} MB)")